# Importamos las librerias a utilizar

In [2]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00


In [3]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 4.0 MB/s eta 0:00:00


# Cargamos el PDF, leemos cada página y almacenamos el texto en una variable

In [5]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Cargar el PDF desde ,las carpetas de Drive
pdf_path = "/content/el principito.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Extraer el texto de cada página
text = "\n".join([doc.page_content for doc in documents])

# Segmentamos el texto en chunks, dando la opción de que el texto en cada chunk se pueda solapar

In [6]:
# Dividir en fragmentos (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 200)

chunks = text_splitter.split_text(text)  # revisar como usar .split_docuement para que sea compatible con chromaDB

# Creamos una función la cual, usando un modelo de embedding, me convierta texto a vector de números

In [7]:
from sentence_transformers import SentenceTransformer

def text_to_vector(text):

    # Cargar el modelo de embeddings
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Convertir el texto en un vector numérico
    vector = model.encode(text)

    return vector

# Haciendo uso de text_to_vector, convertimos cada chunk del pdf en vector y lo almacenamos en una lista

In [8]:
chunk_vectors = []
chunk_vectors_len = []
for chunk in chunks:
    vector = text_to_vector(chunk)
    chunk_vectors.append(vector)
    chunk_vectors_len.append(len(vector))

print(len(chunk_vectors))
print(chunk_vectors_len)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

264
[384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 

# Construimos un árbol de tipo Ball Tree, que son más eficientes para vectores de grandes dimensiones

In [16]:
from sklearn.neighbors import BallTree
import numpy as np

def build_ball_tree(vectors):
    tree = BallTree(vectors, leaf_size=30)
    return tree

def search_ball_tree(tree, query_vector, k=3):
    distances, indices = tree.query([query_vector], k=k)
    return distances, indices

tree = build_ball_tree(chunk_vectors)

# Prueba; Ingresamos una frase u oración, la convertimos a su versión de vector por medio del modelo de embedding y realizamos la busqueda en el árbol, al final imprimimos el texto ingresado y el texto resultante que tiene mayor similitud

In [20]:
text = input("Ingrese una frase: ")

query_vector = text_to_vector(text)
distances, indices = search_ball_tree(tree, query_vector, k=1)

print("\nFrase ingresada: " + text + "\n")
print("Frase encontrada: " + chunks[indices[0][0]])

Ingrese una frase: hasta	los	libros	para	niños.	Tengo	una	tercera	excusa:	esta

Frase ingresada: hasta	los	libros	para	niños.	Tengo	una	tercera	excusa:	esta

Frase encontrada: hasta	los	libros	para	niños.	Tengo	una	tercera	excusa:	esta	persona	mayor	vive
en	Francia,	donde	pasa	hambre	y	frío.	Verdaderamente	necesita	consuelo.	Si
todas	esas	excusas	no	bastasen,	bien	puedo	dedicar	este	libro	al	niño	que	una
vez	fue	esta	persona	mayor.	Todos	los	mayores	han	sido	primero	niños.	(Pero
pocos	lo	recuerdan).	Corrijo,	pues,	mi	dedicatoria:
A	LEON	WERTH	CUANDO	ERA	NIÑO
	
	
I
	
Cuando	yo	tenía	seis	años	vi	en	un	libro	sobre	la	selva	virgen	que	se


#Referecia

como referencia vamos a tener el desempeño de *chromaDB*

Se debe revisar desde aqui has abajo por que no esta funcionando bien el codigo

In [ ]:
pip install langchain chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 6.6 MB/s eta 0:0

In [ ]:
!pip3 install langchain-community

In [ ]:
!pip install -qU chromadb langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 79.1 MB/s eta 0:00:00


In [ ]:
pip install langchain-community transformers sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 843.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 69.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [ ]:
!pip install langchain_huggingface

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


model_name = "sentence-transformers/all-MiniLM-L12-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)


vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={'k': 1})

AttributeError: 'str' object has no attribute 'page_content'